# LSTM演示

RNN/LSTM应用场景是**序列数据**，比如NLP里的翻译，**化学中的反应机理(化学反应步骤)、基因表达(分析蛋白质和核酸序列)、药物发现(药物作用的靶点序列)、材料合成(合成步骤序列)、化学动力学(反应速率和时间的关系)、环境监测(污染物浓度变化)等。**

本演示使用LSTM在MNIST数据集上进行了分类测试，最后准确率可达97%+。 演示中把28x28的MNIST手写数字图像看作**序列数据**(序列：长度为28 -- 时间步数/样本数, 维度为28 --  特征数)。   
注：在自然语言处理中，一个句子可以被分割成多个单词或字符，每个单词或字符对应一个时间步；在时间序列分析中，每个时间步可能对应一个时间点上的观测值。而每个单词通过一个词嵌入（word embedding）层被转换为一个固定维度的向量，那么这个向量的维度就是输入维度。

RNN由以下部分组成：
* 将28维输入映射到128维隐藏层的线性层  
* 一种中间递归神经网络（LSTM）  
* 一个完全连接的层，将128维输入映射到10维类标签向量。

**依赖包**

* torch
* torchvision
* ...

**知识点**

* [梯度爆炸](https://www.cnblogs.com/emanlee/p/17068162.html)
* [梯度裁剪](https://blog.51cto.com/u_16175499/6823107)

## 0 基本包导入

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch import Tensor
import torch.nn.functional as F

import pdb
import math


# 判断当前机器是否有可用的GPU，有则把计算放到GPU上运行
cuda = True if torch.cuda.is_available() else False
# 统一张量类型：有GPU时使用cuda版FloatTensor，否则使用CPU版（覆盖上面的导入，后续直接用Tensor即可）
Tensor = torch.cuda.FloatTensor if cuda else torch.FloatTensor

# 固定随机数种子，保证每次运行实验结果可复现
torch.manual_seed(125)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(125)  # GPU上的随机数种子也要固定

## 1 数据集加载

**注：**此处把root设置成'../data'(上一层的data目录)而不是'./data'(本层的data目录)，这样如果其它演示代码也使用相同设置，则在使用MNIST数据集时，就可以不用再下载。可以把ANN演示中的data目录(包括已下载的MNIST数据)直接移到上一层，就可以不用再次下载了。

In [2]:
batch_size = 100  # 每个小批量的样本数
# 加载MNIST手写数字训练集：28x28的图像后续会被当作"28个时间步、每步28维"的序列喂给LSTM
train_dataset = dsets.MNIST(root='../data',
                            train=True,  # 指定为训练集
                            transform=transforms.ToTensor(),  # 把PIL图像转成Tensor
                            download=True)  # 本地不存在时自动下载

test_dataset = dsets.MNIST(root='../data',
                           train=False,  # 指定为测试集
                           transform=transforms.ToTensor())

# 训练集加载器：shuffle=True表示每个epoch随机打乱样本顺序，让训练更稳定
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

# 测试集加载器：评估阶段无需打乱，保证结果可对比
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

## 2 构建模型

### 2.1 LSTM Cell

In [5]:
import torch
import torch.nn as nn
import math

class LSTMCell(nn.Module):
    """
    实现了Hochreiter和Schmidhuber的长短期记忆（LSTM）单元。
    LSTM是一种特殊的循环神经网络（RNN）单元，能够学习长期依赖关系。
    """

    def __init__(self, input_size, hidden_size, bias=True):
        super(LSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bias = bias
        
        # 定义从输入到隐藏状态的线性变换
        self.x2h = nn.Linear(input_size, 4 * hidden_size, bias=bias)
        # 定义从隐藏状态到隐藏状态的线性变换
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size, bias=bias)
        
        # 初始化参数
        self.reset_parameters()

    def reset_parameters(self):
        """
        初始化网络参数，使用均匀分布。
        """
        std = 1.0 / math.sqrt(self.hidden_size)
        for w in self.parameters():
            w.data.uniform_(-std, std)

    def forward(self, x, hidden):
        """
        前向传播方法，计算LSTM单元的输出。
        
        参数:
        x -- 输入数据  shape=[batch_size, input_dim]
        hidden -- 包含前一个时间步的隐藏状态和细胞状态的元组
        
        返回:
        新的隐藏状态和细胞状态
        """
        hx, cx = hidden  # 解包前一个时间步的隐藏状态和细胞状态
        
        # 计算所有门控的总和
        gates = self.x2h(x) + self.h2h(hx)
        
        # 将门控信号分割为四个部分：输入门、遗忘门、细胞门和输出门
        ingate, forgetgate, cellgate, outgate = gates.chunk(4, 1)
        
        # 应用sigmoid函数激活输入门和输出门，使用tanh函数激活细胞门
        ingate = torch.sigmoid(ingate)
        forgetgate = torch.sigmoid(forgetgate)
        outgate = torch.sigmoid(outgate)
        cellgate = torch.tanh(cellgate)
        
        # 计算新的细胞状态
        cy = (forgetgate * cx) + (ingate * cellgate)
        
        # 计算新的隐藏状态
        hy = outgate * torch.tanh(cy)
        
        return (hy, cy)


**示例使用LSTM Cell**

In [11]:

# 创建LSTM单元实例，输入大小为10，隐藏层大小为20
lstm_cell = LSTMCell(10, 20)

# 假设输入数据x和初始隐藏状态
x = torch.randn(1, 10)  # 假设批大小为1
hx = torch.zeros(1, 20)  # 初始隐藏状态
cx = torch.zeros(1, 20)  # 初始细胞状态

# 前向传播
hy, cy = lstm_cell(x, (hx, cx))

### 2.2 LSTM Model

In [10]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, bias=True):
        """
        初始化LSTM模型。

        参数:
        input_dim (int): 输入数据的维度。
        hidden_dim (int): LSTM隐藏层的维度。
        layer_dim (int): LSTM层的数量。
        output_dim (int): 输出层的维度。
        bias (bool): 是否使用偏置项。
        """
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        # 定义LSTM层
        # （复用上面的自定义LSTMCell，时间步循环在forward里手动展开；layer_dim仅作占位参数）
        self.lstm = LSTMCell(input_dim, hidden_dim, bias)

        # 定义全连接层，将LSTM的输出映射到最终的输出维度
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        前向传播方法。

        参数:
        x (Tensor): 输入数据，形状为 (batch_size, sequence_length, input_dim)。

        返回:
        Tensor: 模型的输出。
        """
        # 根据是否使用GPU来初始化隐藏状态和细胞状态
        if torch.cuda.is_available():
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).cuda()
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).cuda()
        else:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)

        # 初始化输出列表，用于存储每个时间步的输出
        outs = []

        # 初始化细胞状态和隐藏状态（从零开始）
        cn = c0[0,:,:]
        hn = h0[0,:,:]

        # 对输入序列的每个时间步进行迭代
        for seq in range(x.size(1)):
            # 获取当前时间步的输入
            current_input = x[:, seq, :]  # shape = [B, I]
            # 通过LSTM层传递当前输入，并更新隐藏状态和细胞状态
            hn, cn = self.lstm(current_input, (hn, cn))
            # 将当前时间步的隐藏状态添加到输出列表
            outs.append(hn)

        # 获取最后一个时间步的隐藏状态（它"读过"整张图，包含全序列信息，用于分类）
        out = outs[-1].squeeze()

        # 通过全连接层传递最后一个时间步的隐藏状态，得到最终的输出
        out = self.fc(out)
        return out

## 3 实例化模型

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义模型参数
input_dim = 28       # 输入数据的维度
hidden_dim = 128    # LSTM隐藏层的维度
layer_dim = 2       # LSTM层的数量，这里从1层改为2层
output_dim = 10     # 输出层的维度

# 实例化LSTM模型
model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim)

# 如果GPU可用，则将模型移动到GPU上
if torch.cuda.is_available():
    model.cuda()


## 4 损失函数与优化器
使用交叉熵损失函数（常用于多分类问题）。用随机梯度下降（SGD）作为优化器，并设置学习率。

In [8]:
# 交叉熵损失：适用于10分类任务（内部含softmax，与模型输出的10维logits配合使用）
criterion = nn.CrossEntropyLoss()
learning_rate = 0.1  # 设置学习率
optimizer = optim.SGD(model.parameters(), lr=learning_rate)  # 实例化SGD优化器

## 5 模型训练
**训练过程逻辑解释：**  

**1、初始化和设置：**  
* 定义序列的维度seq_dim。  
* 初始化损失值列表loss_list和迭代次数计数器iter。

**2、训练过程：**  
* 遍历指定的训练轮次num_epochs。  
* 在每个轮次中，遍历训练数据加载器train_loader。  
* 调整输入图像的形状，并根据是否可用的GPU来移动数据到相应的设备上。  
* 清除梯度。  
* 前向传播，获取模型输出。  
* 计算损失。  
* 反向传播，计算梯度。  
* 更新模型参数。

**3、评估过程：**  
* 每100次迭代，评估模型在测试集上的性能。  
* 遍历测试数据加载器test_loader。  
* 调整输入图像的形状，并根据是否可用的GPU来移动数据到相应的设备上。  
* 前向传播，获取模型输出。  
* 获取预测结果，并计算准确率。  
* 打印当前迭代次数、损失和准确率。  

In [ ]:
import torch


n_iters = 6000  # 原计划的训练总迭代数（此处仅作参考，实际训练轮数由下面的num_epochs决定）
# num_epochs = int(n_iters / (len(train_dataset) / batch_size))
num_epochs = 5  #  此处纯粹为了演示设置了num_epochs=5


# 定义序列的维度：每张28x28图像按行拆成28个时间步，每个时间步输入1行28维
seq_dim = 28

# 初始化损失值列表
loss_list = []
# 初始化迭代次数计数器
iter = 0

# 遍历指定的训练轮次
for epoch in range(num_epochs):
    # 遍历训练数据加载器
    for i, (images, labels) in enumerate(train_loader):
        # 调整输入图像的形状，并根据是否可用的GPU来移动数据到相应的设备上
        if torch.cuda.is_available():
            images = images.view(-1, seq_dim, input_dim).cuda()
            labels = labels.cuda()
        else:
            images = images.view(-1, seq_dim, input_dim)
            labels = labels

        # 清除梯度（上一次迭代计算的梯度若不清零会累加）
        optimizer.zero_grad()

        # 前向传播，获取模型输出
        outputs = model(images)

        # 计算损失
        loss = criterion(outputs, labels)

        # 如果GPU可用，将损失移到GPU上
        if torch.cuda.is_available():
            loss = loss.cuda()

        # 反向传播，计算梯度
        loss.backward()

        # 对模型进行裁剪，防止模型梯度爆炸
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)

        # 更新模型参数
        optimizer.step()

        # 记录损失值
        loss_list.append(loss.item())
        # 更新迭代次数
        iter += 1

        # 每100次迭代打印一次训练信息
        if iter % 100 == 0:
            # 初始化正确预测和总预测的数量
            correct = 0
            total = 0

            # 遍历测试数据加载器（边训练边在测试集上评估准确率）
            for images, labels in test_loader:
                # 调整输入图像的形状，并根据是否可用的GPU来移动数据到相应的设备上
                if torch.cuda.is_available():
                    images = images.view(-1, seq_dim, input_dim).cuda()
                else:
                    images = images.view(-1, seq_dim, input_dim)

                # 前向传播，获取模型输出
                outputs = model(images)

                # 获取预测结果（取10个类别中得分最大的作为预测类别）
                _, predicted = torch.max(outputs.data, 1)

                # 更新总预测数量
                total += labels.size(0)

                # 更新正确预测数量
                if torch.cuda.is_available():
                    correct += (predicted.cpu() == labels.cpu()).sum()
                else:
                    correct += (predicted == labels).sum()

            # 计算准确率
            accuracy = 100 * correct / total

            # 打印当前迭代次数、损失和准确率
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))

## 作业：
**以上只演示了了LSTM模型及训练（只有简化版的训练，没有early stopping及best model）。**
1) 试使用optim.Adam优化器时，看准确率有何变化？
2) 按ANN演示里介绍的训练及预测的方式方法重写训练及预测代码。
3) 思考：如果没有作梯度裁剪(注释掉torch.nn.utils.clip_grad_norm_所在行代码)，看准确率有何变化？为什么此处基本变化不大？
4) **思考：为什么在"4.1_ann/mnist/ANN.ipynb"的"预测"时会有"load_state_dict(best_model)"这一步？ 因为在实际中，训练与预测是分离的步骤，会拿着训练好的模型预测当前新生成的数据，因此需要把训练的模型保存(成模型文件)，在需要预测时时再加载出来使用。**